1.Difference between "Love" and "love": In NLP, computers see these as different tokens because of their ASCII values (L=76, l=108). Without lowercasing, a model would treat them as two distinct words, splitting the "importance" or "weight" of the word.

2.What happens if stopwords are not removed? The feature matrix becomes "noisy" and "sparse." High-frequency words like "the" or "is" dominate the dataset, potentially overshadowing the meaningful words that actually define the sentiment or topic.

3.Harsh Scenarios for Stopword Removal:

  i)Sentiment Analysis: Removing "not" from "I am not happy" turns it into "I am happy," completely flipping the sentiment.

  ii)Search Queries/Question Answering: In a query like "To be or not to be," removing stopwords leaves nothing, losing the entire context of the famous quote.

4.Stemming vs. Lemmatization: Stemming is a rule-based "chopping" (e.g., studies -> studi) which is fast but can create non-dictionary words. Lemmatization uses a dictionary to find the morphological root (e.g., studies -> study), which is more accurate but computationally slower.

In [1]:
import re
import string
import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')

def preprocess_text(text):
    # 1. Handle Empty Strings/Edge Cases (Task 7)
    if not text or str(text).strip() == "":
        return ""

    # 2. Convert to lowercase
    text = text.lower()

    # 3. Remove URLs and Email-like patterns
    text = re.sub(r'http\S+|www\S+|[\w\.-]+@[\w\.-]+', '', text)

    # 4. Remove Numbers
    text = re.sub(r'\d+', '', text)

    # 5. Handle Repeated Characters (soooo -> so)
    # This regex looks for 3 or more repeated chars and reduces to 2 (to keep words like 'cool')
    text = re.sub(r'(.)\1{2,}', r'\1', text)

    # 6. Remove Punctuation (Except for basic text)
    text = text.translate(str.maketrans('', '', string.punctuation))

    # 7. Remove Extra Spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # 8. Tokenization
    tokens = word_tokenize(text)

    # 9. Remove very short tokens (Keep 'no', 'not')
    meaningful_short_words = {'no', 'not'}
    clean_tokens = [t for t in tokens if len(t) > 2 or t in meaningful_short_words]

    return clean_tokens

# Test the function logic
print(f"Test Result: {preprocess_text('soooo goooood!!! Visit http://test.com')}")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Test Result: ['god', 'visit']


In [2]:
sample_inputs = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product 😍😍",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

all_data = []

print(f"{'Original Text':<35} | {'Cleaned Sentence'}")
print("-" * 70)

for text in sample_inputs:
    tokens = preprocess_text(text)
    clean_sentence = " ".join(tokens)

    # Task 4: Analytics
    num_tokens = len(tokens)
    unique_tokens = len(set(tokens))
    avg_len = sum(len(t) for t in tokens) / num_tokens if num_tokens > 0 else 0

    all_data.append({
        "original": text,
        "tokens": tokens,
        "clean": clean_sentence,
        "count": num_tokens,
        "unique": unique_tokens,
        "avg_len": avg_len
    })

    print(f"{text[:34]:<35} | {clean_sentence}")

# Analysis Questions (Task 4)
# Most noise = Largest difference between raw length and cleaned length
most_noise = max(all_data, key=lambda x: len(x['original']) - len(x['clean']))
print(f"\nSentence with most noise: '{most_noise['original']}'")

Original Text                       | Cleaned Sentence
----------------------------------------------------------------------
Get 100% FREE access now!!!         | get free access now
I absolutely looooved this product  | absolutely loved this product
Worst service ever... 0/10          | worst service ever
Call me at 9876543210               | call
This is THE best course!!!          | this the best course
Visit https://openai.com now!       | visit now
Nooooo this is baaad!!!             | no this bad
OK OK OK I got it                   | got
Win $$$ now!!! Limited offer!!!     | win now limited offer
I am not happy with this            | not happy with this

Sentence with most noise: 'Visit https://openai.com now!'


In [3]:
from collections import Counter

# Flatten all lists of tokens into one big list
all_tokens = [token for item in all_data for token in item['tokens']]

word_freq = Counter(all_tokens)

print("Top 10 Most Frequent Words:", word_freq.most_common(10))
print("Top 5 Least Frequent Words:", word_freq.most_common()[:-6:-1])

Top 10 Most Frequent Words: [('this', 4), ('now', 3), ('get', 1), ('free', 1), ('access', 1), ('absolutely', 1), ('loved', 1), ('product', 1), ('worst', 1), ('service', 1)]
Top 5 Least Frequent Words: [('with', 1), ('happy', 1), ('not', 1), ('offer', 1), ('limited', 1)]


In [4]:
def full_pipeline(text_list):
    results = {
        "tokens": [],
        "clean_sentences": []
    }

    for text in text_list:
        clean_tokens = preprocess_text(text)
        results["tokens"].append(clean_tokens)
        results["clean_sentences"].append(" ".join(clean_tokens))

    return results

# Task 7: Final Error Handling Test
test_cases = ["", "😍😍😍", "123456", "   "]
final_results = full_pipeline(test_cases)

print("Error Handling Results:")
for i, txt in enumerate(test_cases):
    print(f"Input: '{txt}' -> Cleaned: '{final_results['clean_sentences'][i]}'")

Error Handling Results:
Input: '' -> Cleaned: ''
Input: '😍😍😍' -> Cleaned: ''
Input: '123456' -> Cleaned: ''
Input: '   ' -> Cleaned: ''
